In [1]:
import os
import re
from pathlib import Path
from datetime import datetime

# Maths
import numpy as np
import pandas as pd
import seaborn as sns

# Database
from sqlalchemy import Column, Integer, BigInteger, Float, String, Boolean, DateTime, Date, ForeignKey
from sqlalchemy import create_engine, URL
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker

# Helpful imports
from typing import Dict, Any, List
from IPython.display import clear_output, display

# Data scrapping
from bs4 import BeautifulSoup
import requests

# Data scrapping

In [ ]:
res = requests.get("https://opendata.gov.ua/dataset/air_monitor")
soup = BeautifulSoup(res.text, "html.parser")

regex = r"\d{4}-\d{2}-\d{2}"

for item in soup.find_all("div", {"class": "resource-item"}):
    name_block = item.find("div", {"class": "data-resource-name-content"})
    link_tag = name_block.find("a") if name_block else None

    if not link_tag:
        continue

    table_name_raw = link_tag.string or ""
    matches = re.findall(regex, table_name_raw)
    if not matches:
        continue

    table_name = f"{matches[0]}.csv"

    download_tag = item.find("a", {"class": "data-resource-download"}, href=True)
    if not download_tag:
        continue

    table_link = download_tag.get("href")

    file_downloaded = requests.get(table_link)
    print(file_downloaded.status_code, table_name, table_link)

    if file_downloaded.status_code == 200:
        with open(table_name, "wb") as file:
            file.write(file_downloaded.content)

# Data preprocessing

## Data merging

The data was downloaded from the city council website using `scrapper.py` Let's take a closer look at this data

First of all, let's check if all tables have the same columns. If not, you need to perform additional operations to correctly merge the data.

In [2]:
def check_if_columns_same(data_dir: str) -> bool:
    c_set = set()
    files = Path(data_dir).rglob('*.csv')
    for f in files:
        sdf = pd.read_csv(f)
        c_set.add(tuple(sdf.columns))

    return len(c_set) == 1


print(check_if_columns_same("./data"))

True


As you can see, the result of the function is `True`, i.e. all tables have the same columns, which means that the tables can be joined without any problems with column compatibility. To do this, we will define a special function

In [3]:
def combine_tables(data_dir: str) -> pd.DataFrame:
    files = Path(data_dir).rglob('*.csv')
    data_frame = pd.concat(map(pd.read_csv, files))
    return data_frame


df = combine_tables("./data")
df.head()

,stations_id,stations_name,Lat,Long,stations_time,stations_offset,stations_params_id,stations_params_key,stations_params_name,stations_params_localName,stations_params_unit,stations_params_localUnit,stations_params_value,stations_params_cr,stations_params_time,stations_params_offset,stations_params_level
0,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,9.14,1.000000,2022-10-25 13:24:28,0,1
1,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,11.69,1.000000,2022-10-25 13:24:28,0,1
2,767,Соборна 36,49.232859,28.470453,2022-10-25 13:24:28,0,7,CO2,CO₂,CO₂,ppm,мкг/м³,400.00,1.800009,2022-10-25 13:24:28,0,1
3,774,Станція замостя,49.245473,28.493727,2022-10-25 13:25:11,0,33,SDS_P2,PM2.5,Пил 2.5 мкм,ug/m3,мкг/м³,14.04,1.000000,2022-10-25 13:25:11,0,2
4,774,Станція замостя,49.245473,28.493727,2022-10-25 13:25:11,0,34,SDS_P1,PM10,Пил 10 мкм,ug/m3,мкг/м³,19.68,1.000000,2022-10-25 13:25:11,0,1


As you can see, the dataset has been successfully merged, so you can move on to the next stage — removing redundant, duplicate, and conflicting data.

## Data cleaning

Let's look at the missing values

In [4]:
df.isna().sum()

stations_id                        0
stations_name                      0
Lat                                0
Long                               0
stations_time                      0
stations_offset                    0
stations_params_id             41340
stations_params_key                0
stations_params_name               0
stations_params_localName          0
stations_params_unit               0
stations_params_localUnit          0
stations_params_value          86477
stations_params_cr           1765362
stations_params_time               0
stations_params_offset             0
stations_params_level              0
dtype: int64

As you can see, a large amount of data is missing in `stations_params_cr` and a small amount in `stations_params_id` and `stations_params_value`. Since if a parameter does not have an identifier or value, it does not carry any useful information and can be safely removed from the dataset.

In [5]:
def prune_invalid_measurements(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes all records where the measurement value OR the original parameter ID is missing.
    Ensures data integrity before loading into Fact_Measurements.
    """
    df = df.copy()
    initial_count = len(df)

    # Removes rows where at least one of these columns has NaN
    df = df.dropna(subset=['stations_params_value', 'stations_params_id'])

    final_count = len(df)
    dropped_count = initial_count - final_count

    print(f"- Rows removed: {dropped_count}")
    print(f"- Left for processing: {final_count}")

    return df


df = prune_invalid_measurements(df)
df = df.replace(np.nan, None)

- Rows removed: 125924
- Left for processing: 7041122


As you can see, a small portion of the data has been removed (approximately 2%) but the data is now free of empty values.

Next, we will check the data set for stations with the same identifiers but different names to bring them into first normal form.

In [6]:
def identify_duplicate_names(df: pd.DataFrame) -> None:
    # Group data by ID and collect a list of all unique names for each ID
    name_groups = df.groupby('stations_id')['stations_name'].unique()

    # Filter only those groups where the number of unique names is greater than 1
    problematic_stations = name_groups[name_groups.apply(len) > 1]

    # Check for discrepancies
    if problematic_stations.empty:
        print("Name check complete: no discrepancies found.")
        return

    # Print results
    print(f"Found {len(problematic_stations)} IDs with duplicate names:\n")

    for station_id, names in problematic_stations.items():
        # Format the names array into a string for easy reading
        names_list = ", ".join(map(str, names))
        print(f"ID: {station_id} | Names: {names_list}")
        print("-" * 40)


identify_duplicate_names(df)

Found 9 IDs with duplicate names:

ID: 90 | Names: vinnytsia, vinnytsia-90
----------------------------------------
ID: 246 | Names: vinnytsia, vinnytsia-246
----------------------------------------
ID: 256 | Names: vinnytsia, vinnytsia-256
----------------------------------------
ID: 271 | Names: vinnytsia, vinnytsia-271
----------------------------------------
ID: 274 | Names: vinnytsia, vinnytsia-274
----------------------------------------
ID: 281 | Names: vinnytsia, vinnytsia-281
----------------------------------------
ID: 315 | Names: vinnytsia, vinnytsia-315
----------------------------------------
ID: 767 | Names: Соборна 36, Хмельницьке шосе 27
----------------------------------------
ID: 1183 | Names: Вишенька, Славне
----------------------------------------


As you can see, some of the stations have conflicting names. To fix this, an additional function is needed.

In [7]:
def remap_conflicting_names(data: pd.DataFrame) -> pd.DataFrame:
    stations_mapping = {
        256: "vinnytsia-256",
        281: "vinnytsia-281",
        315: "vinnytsia-315",
        90: "vinnytsia-90",
        271: "vinnytsia-271",
        767: "Соборна 36",
        1183: "Вишенька",
        246: "vinnytsia-246",
        274: "vinnytsia-274",
    }
    mask = df['stations_id'].isin(stations_mapping.keys())
    data.loc[mask, 'stations_name'] = data.loc[mask, 'stations_id'].map(stations_mapping)
    return data


df = remap_conflicting_names(df)
identify_duplicate_names(df)

Name check complete: no discrepancies found.


Let's check if all coordinates intersect and have the same origin

In [8]:
def check_coordinate_consistency(df: pd.DataFrame) -> None:
    # Group by ID and count the number of unique latitude and longitude values
    stats = df.groupby('stations_id')[['Lat', 'Long']].nunique()

    # Filter only those IDs where the number of unique coordinates > 1
    problematic_mask = (stats['Lat'] > 1) | (stats['Long'] > 1)
    problematic_ids = stats[problematic_mask].index

    # If no problematic stations are found - exit
    if problematic_ids.empty:
        print("Check complete: all stations have consistent coordinates.")
        return

    # Print details for each problematic station
    print(f"Found {len(problematic_ids)} stations with conflicting coordinates:\n")

    for sid in problematic_ids:
        # Get unique coordinate values for this specific ID
        unique_lats = df.loc[df["stations_id"] == sid, "Lat"].unique()
        unique_longs = df.loc[df["stations_id"] == sid, "Long"].unique()

        print(f"Station ID: {sid}")
        print(f"  - Unique Lat:  {unique_lats}")
        print(f"  - Unique Long: {unique_longs}")
        print("-" * 40)


check_coordinate_consistency(df)

Found 10 stations with conflicting coordinates:

Station ID: 92
  - Unique Lat:  [49.22665528295114 49.227702145800045]
  - Unique Long: [28.44795585 28.44476567]
----------------------------------------
Station ID: 256
  - Unique Lat:  [49.20485607380781 49.20875249539933]
  - Unique Long: [28.5288355  28.52855327]
----------------------------------------
Station ID: 767
  - Unique Lat:  [49.232859 '49.232859' 49.233663]
  - Unique Long: [28.470453 28.438175]
----------------------------------------
Station ID: 774
  - Unique Lat:  [49.245473 '49.245473']
  - Unique Long: [28.493727]
----------------------------------------
Station ID: 790
  - Unique Lat:  [49.227009 '49.227009' 49.2270093]
  - Unique Long: [28.418998  28.4189984]
----------------------------------------
Station ID: 1183
  - Unique Lat:  [49.227518 49.3331555]
  - Unique Long: [28.396157  28.5390885]
----------------------------------------
Station ID: 1315
  - Unique Lat:  [49.232957 '49.232957' '49.23327&' 49.23327]

Since there are stations with conflicting coordinates, they need to be processed. For this purpose, a specialized function was developed that determines the most probable coordinates of the station by finding the mode of the coordinates.

In [9]:
def standardize_coordinates(df: pd.DataFrame) -> None:
    """
    Cleans Lat/Long coordinates, converts them to numeric values, rounds them,
    and replaces all values for each station with their mode (most frequent value).
    """

    cols = ['Lat', 'Long']

    for col in cols:
        # Cleaning and conversion to numeric type
        # If there are strings in the column, keep only digits and the dot
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.extract(r'(\d+\.\d+)')[0]

        # Convert to float and round
        df[col] = pd.to_numeric(df[col], errors='coerce').round(5)

    # Determine the mode for each stations_id
    # Create a helper function to get the first mode to avoid errors if there is no mode 
    # for the group (e.g., all values are NaN)
    def get_first_mode(series):
        m = series.mode()
        return m.iloc[0] if not m.empty else None

    df[cols] = df.groupby('stations_id')[cols].transform(get_first_mode)

    print("Coordinates successfully cleaned and standardized by mode.")


standardize_coordinates(df)
check_coordinate_consistency(df)

Coordinates successfully cleaned and standardized by mode.
Check complete: all stations have consistent coordinates.


Next, let's consider units of measurement.

In [10]:
df["stations_params_key"].unique()

<ArrowStringArray>
[                'SDS_P2',                 'SDS_P1',                    'CO2',
     'BME280_temperature',        'BME280_humidity',        'BME280_pressure',
                    'pm0',                   'pm25',                   'pm10',
            'temperature',               'humidity',                 'PMS_P0',
                 'PMS_P2',                 'PMS_P1',     'ZPHS01B=pm1(ug/m3)',
    'ZPHS01B=pm25(ug/m3)',    'ZPHS01B=pm10(ug/m3)',   'AHTx0=temperature(C)',
  'BMP280=temperature(C)', 'ZPHS01B=temperature(C)',     'AHTx0=humidity(Rh)',
   'ZPHS01B=humidity(Rh)',    'BMP280=pressure(Pa)',       'ZPHS01B=co2(ppm)',
    'ZPHS01B=ch2o(mg/m3)',        'ZPHS01B=o3(ppm)',       'ZPHS01B=no2(ppm)',
        'ZPHS01B=co(ppm)',               'pressure',                   'PM10',
                  'PM2.5',             'VOC (H₂CO)',            'Temperature',
               'Humidity',               'Pressure',                  'PM1.0',
                     'CO',       

As you can see, the station parameters, namely what it measures, are completely unstandardized, have different shapes and are not suitable for further work, so they require detailed refinement. For this, 3 auxiliary functions are declared:
- `unify_measurement_units` – processes measurement units
- `resolve_canonical_parameter` – processes canonical measurement names
- `standardize_dimension_attributes` – a connecting function for the two previous ones

In [11]:
def get_canonical_parameters_map() -> Dict[str, str]:
    """
    Full dictionary mapping technical codes to canonical names.
    """
    return {
        # Dust
        'SDS_P1': 'PM10',
        'SDS_P2': 'PM2.5',
        'PMS_P0': 'PM1.0',
        'PMS_P1': 'PM10',
        'PMS_P2': 'PM2.5',
        'PM0': 'PM1.0',
        'PM1': 'PM1.0',
        'PM25': 'PM2.5',
        'PM100': 'PM10',
        'PM1.0': 'PM1.0',
        'PM2.5': 'PM2.5',
        'PM10': 'PM10',

        # Gases
        'CO2': 'CO2',
        'CO': 'CO',
        'NO2': 'NO2',
        'O3': 'O3',
        'O₃': 'O3',
        'NH3': 'NH3',
        'CH2O': 'HCHO',
        'H2CO': 'HCHO',
        'VOC': 'VOC',
        'NO₂': 'NO2',

        # Meteo
        'TEMPERATURE': 'Temperature',
        'HUMIDITY': 'Humidity',
        'PRESSURE': 'Pressure',

        # Radiation
        'RAD': 'Radiation',

        # Specific Sensor
        'A4': 'CO',   # Often CO on specific manufacturer boards
        'E1': 'NO2',  # Often NO2
        'E3': 'O3',   # Often O3
    }


In [12]:
def resolve_canonical_parameter(raw_key: Any) -> str:
    """
    Recognizes the canonical name of the parameter from a technical string of any complexity.
    """
    if pd.isna(raw_key) or str(raw_key).lower() == 'nan':
        return "UNKNOWN"

    # Cleaning vendor names and units in parentheses
    key = str(raw_key).strip()

    # Remove everything up to and including the '=' sign
    if '=' in key:
        key = key.split('=')[-1]

    # Remove data in parentheses '(...)'
    key = re.sub(r'\(.*?\)', '', key)

    # Basic data formatting
    key = key.replace(' ', '').replace('_', '').replace('.', '')
    key_upper = key.upper()
    param_map = get_canonical_parameters_map()

    # Replacement of incorrect values
    return param_map.get(key_upper, key_upper)


In [13]:
def unify_measurement_units(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardizes measurement units to a single Unicode format.
    """
    # Replacement dictionary for units
    unit_standardization = {
        'ug/m3': 'µg/m³',
        'мкг/м³': 'µg/m³',
        'ug/m³': 'µg/m³',
        'ppm': 'ppm',
        'ppb': 'ppb',
        '%': '%',
        'Rh': '%',
        '°C': '°C',
        'C': '°C',
        'Pa': 'Pa',
        'hPa': 'hPa',
        'mg/m3': 'mg/m³',
        'uSv/h': 'µSv/h'
    }

    def clean_unit(unit_str):
        if pd.isna(unit_str): return 'unknown'
        u = str(unit_str).strip()
        # Removes parentheses if they came with the data
        u = u.replace('(', '').replace(')', '')
        return unit_standardization.get(u, u)

    df['stations_params_unit'] = df['stations_params_unit'].apply(clean_unit)
    return df

In [14]:
def standardize_dimension_attributes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Orchestrator for the previous functions
    """
    df = df.copy()
    df['stations_params_key'] = df['stations_params_key'].apply(resolve_canonical_parameter)
    df = unify_measurement_units(df)
    df['stations_params_name'] = df['stations_params_name'].fillna(df['stations_params_key'])
    return df


df = standardize_dimension_attributes(df)

## Time recycling

The dataset has exceptional situations where the time is recorded incorrectly, to avoid further errors it should be processed to a common format.

In [15]:
def prepare_dataframe_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocessing the dataframe to ensure all date columns have a common format. 
    Supports ISO8601 and mixed formats.
    """
    date_cols = ['stations_time', 'stations_params_time']
    for col in date_cols:
        # format='mixed' for exceptions 
        # '2022-10-25 13:24:28' and '2023-07-03T12:10:00Z'
        df[col] = pd.to_datetime(df[col], format='mixed', utc=True)
    return df


df = prepare_dataframe_dates(df)

# Creating a database model

Let's look at the data that was obtained as a result of processing, or more precisely, at the fields and their data types.

In [16]:
df.dtypes

stations_id                                int64
stations_name                                str
Lat                                      float64
Long                                     float64
stations_time                datetime64[us, UTC]
stations_offset                            int64
stations_params_id                        object
stations_params_key                          str
stations_params_name                         str
stations_params_localName                    str
stations_params_unit                         str
stations_params_localUnit                    str
stations_params_value                    float64
stations_params_cr                        object
stations_params_time         datetime64[us, UTC]
stations_params_offset                     int64
stations_params_level                      int64
dtype: object

As can be seen from the processed data — most of the objects remain of the object type, that is, text data. In order to work better with them, you should create a database using the standard Data Warehouse schema, namely the "Snowflake" model.

In this case, the following tables will be used:
- **Fact table**
    - Dimensions — combines the rest of the database tables
- **Measurement tables**
    - Parameters — stores variable measurement parameters, such as deviations
    - Stations — stores data about the station and its properties
    - Units — stores data about units of measurement

![](./uml/schema.png)

## Implementing a data schema using SQLalchemy

In [17]:
Base = declarative_base()  # creating the base for defining the following models

### Units table definition

In [18]:
class DimUnit(Base):
    """
    Measurement units table, dimension table
    """
    __tablename__ = 'dim_units'

    unit_key = Column(Integer, primary_key=True, autoincrement=True)
    unit_name = Column(String(64), nullable=False)
    unit_symbol = Column(String(8))
    local_unit_name = Column(String(64))

    # Relationships
    parameters = relationship("DimParameter", back_populates="unit")

### Setting parameter tables

In [19]:
class DimParameter(Base):
    """
    Measurement parameters table, dimension table
    """
    __tablename__ = 'dim_parameters'

    parameter_key = Column(Integer, primary_key=True, autoincrement=True)
    parameter_code = Column(String(64), nullable=False)
    parameter_name = Column(String(64))
    local_name = Column(String(64))
    unit_key = Column(Integer, ForeignKey('dim_units.unit_key'))

    # SCD Type 2 columns
    valid_from = Column(DateTime, nullable=False)
    valid_to = Column(DateTime)
    is_current = Column(Boolean, default=True)

    # Relationships
    unit = relationship("DimUnit", back_populates="parameters")
    measurements = relationship("FactMeasurement", back_populates="parameter")

### Setting stations tables

In [20]:
class DimStation(Base):
    """
    Stations table, dimension table
    """
    __tablename__ = 'dim_stations'

    station_key = Column(Integer, primary_key=True, autoincrement=True)
    station_id = Column(Integer)
    station_name = Column(String(128))
    latitude = Column(Float)
    longitude = Column(Float)
    timezone_offset = Column(Integer)

    # SCD Type 2 columns
    valid_from = Column(DateTime, nullable=False)
    valid_to = Column(DateTime)
    is_current = Column(Boolean, default=True)

    # Relationships
    measurements = relationship("FactMeasurement", back_populates="station")

### Fact table definition

In [21]:
class FactMeasurement(Base):
    """
    Fact table
    """
    __tablename__ = 'fact_measurements'

    measurement_id = Column(BigInteger, primary_key=True, autoincrement=True)

    # Foreign keys
    station_key = Column(Integer, ForeignKey('dim_stations.station_key'), nullable=False)
    parameter_key = Column(Integer, ForeignKey('dim_parameters.parameter_key'), nullable=False)

    # Measurements
    value = Column(Float)
    quality_ratio = Column(Float, nullable=True)
    pollution_level = Column(Integer)

    # Metadata (Time is integrated here)
    measurement_timestamp = Column(DateTime, nullable=False, index=True)
    offset_minutes = Column(Integer)

    # Relationships
    station = relationship("DimStation", back_populates="measurements")
    parameter = relationship("DimParameter", back_populates="measurements")

## Creating the physical model

Note regarding `host` — it should be replaced with 
- `127.0.0.1` for connecting to a local mysql database
- the name of the Docker Compose service if running there
- a link to a remote server if necessary

In [22]:
engine = create_engine(
    URL.create(
        drivername="mysql",
        username=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        host="db",
        port=3306,
        database=os.getenv("MYSQL_DATABASE")
    )
)

In [23]:
Base.metadata.create_all(engine)

# Filling the database with data

Let's look at the head of the dataset again

In [24]:
df.head()

,stations_id,stations_name,Lat,Long,stations_time,stations_offset,stations_params_id,stations_params_key,stations_params_name,stations_params_localName,stations_params_unit,stations_params_localUnit,stations_params_value,stations_params_cr,stations_params_time,stations_params_offset,stations_params_level
0,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28+00:00,0,33,SDSP2,PM2.5,Пил 2.5 мкм,µg/m³,мкг/м³,9.14,1.0,2022-10-25 13:24:28+00:00,0,1
1,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28+00:00,0,34,SDSP1,PM10,Пил 10 мкм,µg/m³,мкг/м³,11.69,1.0,2022-10-25 13:24:28+00:00,0,1
2,767,Соборна 36,49.23286,28.47045,2022-10-25 13:24:28+00:00,0,7,CO2,CO₂,CO₂,ppm,мкг/м³,400.00,1.800009,2022-10-25 13:24:28+00:00,0,1
3,774,Станція замостя,49.24547,28.49373,2022-10-25 13:25:11+00:00,0,33,SDSP2,PM2.5,Пил 2.5 мкм,µg/m³,мкг/м³,14.04,1.0,2022-10-25 13:25:11+00:00,0,2
4,774,Станція замостя,49.24547,28.49373,2022-10-25 13:25:11+00:00,0,34,SDSP1,PM10,Пил 10 мкм,µg/m³,мкг/м³,19.68,1.0,2022-10-25 13:25:11+00:00,0,1


As can be seen — data cannot simply be moved to the database without making any changes. Instead, the whole process should be broken into steps and combined into an ETL pipeline.

The pipeline steps should be as follows:
- Transform and load measurement units
- Transform and load measurement parameters
- Transform and load station data
- Transform and load measurement time data
- Form the fact table

Below are the pipeline steps

## Transformation and loading of measurement units

In [25]:
def transform_and_load_units(df: pd.DataFrame, session: Session) -> Dict[str, int]:
    """
    Extracts unique units from the dataframe and loads them into DimUnits.
    Returns a mapping {unit_symbol: unit_key}.
    """
    # Identify unique measurement units
    unique_units = df[['stations_params_unit', 'stations_params_localUnit']].drop_duplicates()

    unit_map = {}
    for _, row in unique_units.iterrows():
        # Check for the existence of measurement units in the table;
        # necessary to avoid data duplication
        unit = session.query(DimUnit).filter_by(unit_symbol=row['stations_params_unit']).first()
        if not unit:
            unit = DimUnit(
                unit_name=row['stations_params_unit'], # use the symbol as the name
                unit_symbol=row['stations_params_unit'],
                local_unit_name=row['stations_params_localUnit']
            )
            session.add(unit)
            session.flush()
        unit_map[unit.unit_symbol] = unit.unit_key

    return unit_map


## Transformation and loading of measurement parameters

In [26]:
def transform_and_load_parameters(
    df: pd.DataFrame,
    session: Session,
    unit_map: Dict[str, int]
) -> Dict[str, int]:
    """
    Extracts unique parameters and loads them into DimParameters.
    Handles normalization references to DimUnits.
    """
    # Similarly to the previous one, gets unique values
    unique_params = df[[
        'stations_params_key', 'stations_params_name',
        'stations_params_localName', 'stations_params_unit'
    ]].drop_duplicates()

    param_map = {}
    for _, row in unique_params.iterrows():
        param = session.query(DimParameter).filter_by(parameter_code=row['stations_params_key']).first()
        if not param:
            param = DimParameter(
                parameter_code=row['stations_params_key'],
                parameter_name=row['stations_params_name'],
                local_name=row['stations_params_localName'],
                unit_key=unit_map.get(row['stations_params_unit']),
                valid_from=datetime.now(),
                is_current=True
            )
            session.add(param)
            session.flush()
        param_map[param.parameter_code] = param.parameter_key

    return param_map


## Transformation and loading of station data

In [27]:
def transform_and_load_stations(df: pd.DataFrame, session: Session) -> Dict[int, int]:
    """
    Extracts unique stations and loads them into DimStations.
    Returns a mapping {business_station_id: surrogate_station_key}.
    """
    unique_stations = df[['stations_id', 'stations_name', 'Lat', 'Long', 'stations_offset']].drop_duplicates()

    station_map = {}
    for _, row in unique_stations.iterrows():
        station = session.query(DimStation).filter_by(station_id=row['stations_id'], is_current=True).first()
        if not station:
            station = DimStation(
                station_id=row['stations_id'],
                station_name=row['stations_name'],
                latitude=row['Lat'],
                longitude=row['Long'],
                timezone_offset=row['stations_offset'],
                valid_from=datetime.now(),
                is_current=True
            )
            session.add(station)
            session.flush()
        station_map[station.station_id] = station.station_key

    return station_map

## Fact table formation

In [28]:
def load_fact_measurements(
    df: pd.DataFrame,
    session: Session,
    station_map: Dict[int, int],
    param_map: Dict[str, int],
    batch_size: int = 5000,
) -> None:
    """
    Iterates through the dataframe and populates the Fact_Measurements table.
    """
    batch_buffer: List[FactMeasurement] = []

    n_added = 0
    df_size = len(df['stations_params_value'])
    for index, row in df.iterrows():
        ts = row['stations_params_time']
        if pd.isna(ts):
            continue

        # Getting surrogate keys from maps
        s_key = station_map.get(row['stations_id'])
        p_key = param_map.get(row['stations_params_key'])

        # Skip record if keys are not in the dictionaries
        if s_key is None or p_key is None:
            continue

        # Creating a model instance
        fact = FactMeasurement(
            station_key=s_key,
            parameter_key=p_key,
            value=row['stations_params_value'],
            quality_ratio=row['stations_params_cr'],
            pollution_level=row['stations_params_level'],
            measurement_timestamp=ts,
            offset_minutes=row['stations_params_offset']
        )
        batch_buffer.append(fact)

        # When the buffer reaches the limit, perform a batch upload
        if len(batch_buffer) >= batch_size:
            session.bulk_save_objects(batch_buffer)
            session.commit()

            # Clear the buffer and free memory from objects in the session
            batch_buffer.clear()
            session.expunge_all()
            n_added += batch_size
            print(f"\rAdded: {n_added}/{df_size}", end="")

    # Loading the remaining data that did not fit into the last full batch
    if batch_buffer:
        session.bulk_save_objects(batch_buffer)
        session.commit()
        session.expunge_all()
        batch_buffer.clear()


## Running the pipeline

Pipeline definition

In [29]:
def run_pipeline(df: pd.DataFrame, session: Session) -> None:
    """
    Orchestrates the transformation and loading process.
    """
    try:
        print("Processing measurement units...")
        u_map = transform_and_load_units(df, session)

        print("Processing measurement parameters...")
        p_map = transform_and_load_parameters(df, session, u_map)

        print("Processing stations...")
        s_map = transform_and_load_stations(df, session)

        print("Loading fact table...")
        load_fact_measurements(df, session, s_map, p_map)

        print("\nETL pipeline loaded.")
    except Exception as e:
        session.rollback()
        print(f"Error during ETL: {e}")
        raise e

Creating a database interaction session

In [30]:
Session = sessionmaker(engine)
session = Session()

Running the pipeline

In [31]:
run_pipeline(df, session)

Processing measurement units...
Processing measurement parameters...
Processing stations...
Loading fact table...
Added: 7040000/7041122
ETL pipeline loaded.
